# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 0          # <-- ENTER YOUR FULL ROLL NUMBER
NAME = ""            # <-- ENTER YOUR NAME

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
# Q1 — Preprocessing
# Run the official process_ticket() on every ticket.
processed = {row.Ticket_ID: process_ticket(row.Ticket_Text)
             for row in df.itertuples(index=False)}

q1_rows = []
for ticket_id, p in processed.items():
    q1_rows.append({
        "Ticket_ID": ticket_id,
        "sentences": len(p["sentences"]),
        "tokens": len(p["tokens"]),
        "punctuation_tokens": len(p["punct"]),
        "stopwords": len(p["stops"]),
        "unique_tokens": len(set(p["tokens"])),
        "final_tokens": len(p["final"])
    })

q1_table = pd.DataFrame(q1_rows)

# The assignment asks for a TOTAL row for count-based columns.
# Unique-token counts are not additive, so the TOTAL row uses the
# corpus-wide number of unique tokens; the other columns are summed.
total_row = {
    "Ticket_ID": "TOTAL",
    "sentences": q1_table["sentences"].sum(),
    "tokens": q1_table["tokens"].sum(),
    "punctuation_tokens": q1_table["punctuation_tokens"].sum(),
    "stopwords": q1_table["stopwords"].sum(),
    "unique_tokens": len(set(t for p in processed.values() for t in p["tokens"])),
    "final_tokens": q1_table["final_tokens"].sum()
}
q1_table = pd.concat([q1_table, pd.DataFrame([total_row])], ignore_index=True)
display(q1_table)

# Show the actual material for one selected ticket (the first focus ticket).
selected_id = FOCUS[0]
selected_text = df.loc[df["Ticket_ID"].eq(selected_id), "Ticket_Text"].iloc[0]
selected = processed[selected_id]

print(f"\nSelected ticket: {selected_id}")
print("Raw text:", selected_text)
print("\nActual tokens:")
print(selected["tokens"])
print("\nPunctuation tokens:")
print(selected["punct"])
print("\nStopwords:")
print(selected["stops"])
print("\nFinal tokens:")
print(selected["final"])

# Observation: identify contraction fragments/non-word-looking final tokens.
wordlike = set(w.lower() for w in [
    "the","and","or","to","of","a","an","is","was","were","be","been",
    "can","could","do","did","not","for","on","in","with","my","it","i"
])
non_wordish = sorted({
    t for p in processed.values() for t in p["final"]
    if t in {"n't", "'s", "'m", "'re", "'ve", "'d", "'ll", "ca", "wo"}
})
print("\nNon-word-looking final tokens found:", non_wordish)
print("They come from word_tokenize() splitting contractions (for example, can't -> ca + n't and it's -> it + 's);")
print("the stopword removal removes common stopwords but does not reconstruct those split contraction pieces.")


*Your answer:*

The observation is dataset-dependent. In the official word_tokenize() pipeline, contractions are split into pieces such as `ca` + `n't` and `it` + `'s`. Stopword removal does not join these pieces back together, so some contraction fragments can remain in the final-token list even though they are not independent dictionary words.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
# Q2 — Regular expressions
import re

raw_texts = df["Ticket_Text"].tolist()
all_raw = "\n".join(raw_texts)

# One raw-text string is used for each extraction.
# Numbers: hexadecimal values, comma-separated integers, decimals, and
# multi-part versions such as 2.0.1 are retained as one match.
number_pattern = r"(?<![\w.])(?:0x[0-9A-Fa-f]+|\d[\d,]*(?:\.\d+)*)(?![\w.])"
long_word_pattern = r"\b[A-Za-z]{6,}\b"
capitalized_pattern = r"\b[A-Z][A-Za-z]*\b"
vowel_word_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"

long_words = re.findall(long_word_pattern, all_raw)
numbers = re.findall(number_pattern, all_raw)
capitalized_words = re.findall(capitalized_pattern, all_raw)
vowel_words = re.findall(vowel_word_pattern, all_raw)

print("Words longer than 5 letters:")
print(long_words)

print("\nNumbers (decimals/versions kept whole):")
print(numbers)

print("\nCapitalized words:")
print(capitalized_words)

print("\nWords beginning with a vowel:")
print(vowel_words)

# Replacement patterns.
email_re = r"\b[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+\b"
url_re = r"https?://[^\s<>\"]+|(?<![@\w])www\.[^\s<>\"]+"
phone_re = r"(?<!\w)(?:\+\d{1,3}[\s-]?)?(?:\d{3,5}[\s-]\d{3,5}(?:[\s-]\d{3,5})?|\d{10})(?!\w)"

def redact_identifiers(text):
    # URL first prevents pieces of a URL from being treated as an email.
    text = re.sub(url_re, "<URL>", text)
    text = re.sub(email_re, "<EMAIL>", text)
    text = re.sub(phone_re, "<PHONE>", text)
    return text

print("\nIdentifier replacement on TEST_LINES:")
for line in TEST_LINES:
    print(redact_identifiers(line))

print("\nIdentifier replacement on generated tickets:")
for row in df.itertuples(index=False):
    print(row.Ticket_ID, "->", redact_identifiers(row.Ticket_Text))

# Explicitly verify that the same patterns match the TEST_LINES.
print("\nTEST_LINES extraction check:")
for i, line in enumerate(TEST_LINES, 1):
    print(f"Line {i}:")
    print("  emails:", re.findall(email_re, line))
    print("  urls:", re.findall(url_re, line))
    print("  phones:", re.findall(phone_re, line))
    print("  numbers:", re.findall(number_pattern, line))


*Your answer:*

The notebook computes this from the generated dataset by comparing capitalized matches with the set of words that occur at the beginning of a sentence. Sentence-initial capitalization is therefore not necessarily evidence that the word is a proper name or acronym.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
# Q3 — Custom tokenizer
from nltk.tokenize import RegexpTokenizer

# One RegexpTokenizer pattern. Alternatives are ordered from the most
# structured tokens to ordinary words/numbers.
TOKEN_PATTERN = r"""
https?://[^\s<>\"]+|www\.[^\s<>\"]+
|[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+
|\+\d{1,3}[\s-]?\d{3,5}[\s-]\d{3,5}(?:[\s-]\d{3,5})?
|\d{3,5}[\s-]\d{3,5}(?:[\s-]\d{3,5})?
|\d{10}
|\d+(?:\.\d+)+
|[A-Za-z]+(?:-[A-Za-z]+)+
|[A-Za-z]+(?:'[A-Za-z]+|n't)
|[A-Za-z]+
|\d+(?:,\d+)*
"""

# Remove whitespace/comments from the verbose pattern because RegexpTokenizer
# expects a single regex expression.
TOKEN_PATTERN = re.sub(r"\s+", "", TOKEN_PATTERN)
_custom_tokenizer = RegexpTokenizer(TOKEN_PATTERN, flags=re.IGNORECASE)

def my_tokenize(text):
    return _custom_tokenizer.tokenize(text)

print("TEST_LINES — my_tokenize():")
for i, line in enumerate(TEST_LINES, 1):
    print(f"\nLine {i}:")
    print(my_tokenize(line))

# Compare TEST_LINES and every generated ticket with word_tokenize().
def compare_tokens(text):
    wt = word_tokenize(text)
    mt = my_tokenize(text)
    return wt, mt

print("\nTEST_LINES — comparison with word_tokenize():")
for i, line in enumerate(TEST_LINES, 1):
    wt, mt = compare_tokens(line)
    print(f"\nLine {i}")
    print("word_tokenize:", wt)
    print("my_tokenize  :", mt)

ticket_comparisons = []
for row in df.itertuples(index=False):
    wt, mt = compare_tokens(row.Ticket_Text)
    ticket_comparisons.append({
        "Ticket_ID": row.Ticket_ID,
        "word_tokenize_tokens": wt,
        "my_tokenize_tokens": mt,
        "different": wt != mt
    })

ticket_comparison_df = pd.DataFrame(ticket_comparisons)
print("\nTickets tokenized differently:")
display(ticket_comparison_df[["Ticket_ID", "different"]])
print("Count:", int(ticket_comparison_df["different"].sum()), "of", len(ticket_comparison_df))

# Show concrete difference types.
examples = []
for row in df.itertuples(index=False):
    wt, mt = compare_tokens(row.Ticket_Text)
    if wt != mt:
        wt_set, mt_set = set(wt), set(mt)
        examples.append({
            "Ticket_ID": row.Ticket_ID,
            "word_tokenize_only": sorted(wt_set - mt_set),
            "my_tokenize_only": sorted(mt_set - wt_set)
        })
    if len(examples) >= 5:
        break

print("\nConcrete examples of differences:")
display(pd.DataFrame(examples))

print("""
Why they differ:
1. word_tokenize() separates hyphenated forms such as Wi-Fi/state-of-the-art,
   while my_tokenize() keeps the whole hyphenated form as one token.
2. word_tokenize() splits URLs/emails into several pieces around punctuation,
   while my_tokenize() keeps each URL/email as one token.
3. word_tokenize() separates contraction pieces such as "isn't" into "is" + "n't",
   while my_tokenize() keeps "isn't" as one token.
4. The custom tokenizer also keeps decimals/versions such as 95.5 and 2.0.1
   as one token.
""")


*Your answer:*

The notebook prints the exact count from the generated tickets. The main differences come from contractions, hyphenated words, URLs/emails, phone numbers, and decimals/versions, because the custom tokenizer deliberately keeps these structured forms together.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Q4 — Stemming and lemmatization
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

unique_final = sorted(set(t for p in processed.values() for t in p["final"]))

porter = PorterStemmer()
lancaster = LancasterStemmer()

# Choose a suffix that is visibly represented by at least two words in this dataset.
suffix_candidates = ["ing", "ed", "ly", "s"]
suffix_evidence = {
    suf: sorted(w for w in unique_final if w.endswith(suf) and len(w) > len(suf) + 2)
    for suf in suffix_candidates
}
chosen_suffix = next((s for s in suffix_candidates if len(suffix_evidence[s]) >= 2), "ing")
regexp_stemmer = RegexpStemmer(regex=chosen_suffix + r"$", min=3)

print("Regexp Stemmer suffix:", repr(chosen_suffix))
print("Dataset words supporting the choice:", suffix_evidence.get(chosen_suffix, []))
print("Justification: the selected suffix occurs in at least two words in this generated dataset,")
print("so the rule is based on observed dataset vocabulary rather than an arbitrary suffix.")

lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

# POS-tag all unique final tokens.
pos_tags = dict(nltk.pos_tag(unique_final))

rows = []
for word in unique_final:
    wn_pos = penn_to_wordnet(pos_tags[word])
    rows.append({
        "word": word,
        "POS": pos_tags[word],
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "WordNet": lemmatizer.lemmatize(word, pos=wn_pos)
    })

stem_df_all = pd.DataFrame(rows)
stem_df = stem_df_all[
    (stem_df_all["Porter"] != stem_df_all["Lancaster"]) |
    (stem_df_all["Porter"] != stem_df_all["Regexp"]) |
    (stem_df_all["Porter"] != stem_df_all["WordNet"]) |
    (stem_df_all["Lancaster"] != stem_df_all["Regexp"]) |
    (stem_df_all["Lancaster"] != stem_df_all["WordNet"]) |
    (stem_df_all["Regexp"] != stem_df_all["WordNet"])
].copy()

print("\nWords for which at least one method differs:")
display(stem_df)

# Automatically identify illustrative examples for the observation.
over_candidates = stem_df[
    (stem_df["Porter"].str.len() < stem_df["word"].str.len()) &
    (stem_df["WordNet"] == stem_df["word"])
]
under_candidates = stem_df[
    (stem_df["WordNet"] != stem_df["word"]) &
    (stem_df["Porter"] == stem_df["word"])
]

print("\nObservation examples:")
if len(over_candidates):
    ex = over_candidates.iloc[0]
    print(f"Possible over-stemming: '{ex.word}' -> Porter '{ex.Porter}' while WordNet keeps '{ex.WordNet}'.")
else:
    print("No clear automatic over-stemming candidate was found; inspect the displayed table.")

if len(under_candidates):
    ex = under_candidates.iloc[0]
    print(f"Possible under-stemming: '{ex.word}' remains '{ex.Porter}' with Porter while WordNet gives '{ex.WordNet}'.")
else:
    print("No clear automatic under-stemming candidate was found; inspect the displayed table.")


*Your answer:*

Use the examples printed immediately below the Q4 table. An over-stemming example is one where a stemmer removes too much and produces an unintuitive stem; an under-stemming example is one where related morphology is not reduced to the expected common form. The notebook selects examples from the actual generated vocabulary rather than assuming words that may not occur in your dataset.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Q5 — Corpus statistics
from collections import Counter
from nltk.util import bigrams
import numpy as np
import matplotlib.pyplot as plt

S1 = [t for p in processed.values() for t in p["tokens"]]
S2 = [t for t in S1 if not is_punct(t)]
S3 = [t for p in processed.values() for t in p["final"]]
S4 = [porter.stem(t) for t in S3]

def corpus_stats(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0
    return total, unique, ttr

stats = pd.DataFrame(
    [corpus_stats(S1), corpus_stats(S2), corpus_stats(S3), corpus_stats(S4)],
    index=["S1: all tokens", "S2: without punctuation", "S3: final tokens", "S4: Porter-stemmed S3"],
    columns=["total_tokens", "unique_tokens", "TTR"]
)
display(stats)

print("\nTop 10 words of S1:")
top10_S1 = Counter(S1).most_common(10)
print(top10_S1)

print("\nTop 10 words of S3:")
top10_S3 = Counter(S3).most_common(10)
print(top10_S3)

# Zipf-style plot for S3.
freqs = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)

plt.figure(figsize=(7, 5))
plt.loglog(ranks, freqs, marker="o", linestyle="none")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("S3: log(rank) vs log(frequency)")
plt.grid(True, alpha=0.25)
plt.show()

print("\nTop 5 bigrams of S1:")
print(Counter(bigrams(S1)).most_common(5))

print("\nTop 5 bigrams of S3:")
print(Counter(bigrams(S3)).most_common(5))

# Short interpretations based on the actual corpus.
s1_ttr = stats.loc["S1: all tokens", "TTR"]
s4_ttr = stats.loc["S4: Porter-stemmed S3", "TTR"]
direction = "decreases" if s4_ttr < s1_ttr else "increases" if s4_ttr > s1_ttr else "stays the same"

print("\nQ5 interpretation:")
print(f"TTR {direction} from S1 ({s1_ttr:.4f}) to S4 ({s4_ttr:.4f}) because Porter stemming maps related word forms")
print("to common stems, which changes the number of distinct types relative to the number of tokens.")
print("Punctuation and stopword removal also change the token pool before S4 is formed.")


*Your answer:*

TTR changes because S1 contains every token, including punctuation and stopwords, whereas S4 removes punctuation/stopwords and then applies Porter stemming. Stemming merges some related word forms into the same stem, reducing vocabulary diversity relative to the token count; the notebook prints the exact S1 and S4 TTR values for this dataset.

---
## Q6. Interpretation

The assignment's final interpretation questions are answered below from the generated dataset and the outputs above.


In [ ]:
# Q6 — Interpretation
# (a) TTR
print("(a) TTR change from S1 to S4:")
print(f"S1 TTR = {stats.loc['S1: all tokens', 'TTR']:.4f}")
print(f"S4 TTR = {stats.loc['S4: Porter-stemmed S3', 'TTR']:.4f}")
print("TTR changes because preprocessing removes punctuation/stopwords and Porter stemming")
print("collapses some related surface forms into shared stems.")

# (b) Tokenizer difference
difference_found = None
for row in df.itertuples(index=False):
    wt, mt = compare_tokens(row.Ticket_Text)
    if wt != mt:
        for w in wt:
            if w not in mt and ("-" in w or "'" in w or "@" in w or "." in w):
                difference_found = (row.Ticket_ID, w, wt, mt)
                break
    if difference_found:
        break

if difference_found:
    tid, token, wt, mt = difference_found
    print(f"\n(b) Example: {tid} contains '{token}'.")
    print("word_tokenize() and my_tokenize() treat this structured form differently;")
    print("the custom tokenizer was explicitly designed to preserve contractions, hyphenated")
    print("words, URLs/emails, phone numbers, decimals, and versions.")
else:
    print("\n(b) See the TEST_LINES comparison above for a concrete tokenizer difference.")

# (c) Stemming undesirable result
bad = stem_df[
    (stem_df["Porter"] != stem_df["WordNet"]) &
    (stem_df["Porter"] != stem_df["word"])
]
if len(bad):
    ex = bad.iloc[0]
    print(f"\n(c) Example of an undesirable stemming result: '{ex.word}' -> '{ex.Porter}'.")
    print("Stemming is rule-based and can produce a non-word or lose useful linguistic information.")
else:
    print("\n(c) Inspect the displayed Q4 table for an undesirable stemming result.")

# (d) POS-aware lemmatization
# Prefer a verb/noun example where the lemma differs from the surface form.
lemma_examples = stem_df[
    (stem_df["WordNet"] != stem_df["word"]) &
    (stem_df["POS"].str.startswith(("V", "N")))
]
if len(lemma_examples):
    ex = lemma_examples.iloc[0]
    print(f"\n(d) POS-aware lemmatization example: '{ex.word}' ({ex.POS}) -> '{ex.WordNet}'.")
    print("Because WordNet uses the POS tag, it can map an inflected form to an appropriate dictionary lemma")
    print("instead of merely chopping off a suffix as a stemmer does.")
else:
    print("\n(d) Inspect the Q4 table for a POS-aware lemma example.")
